### The question it should answer
1. Are all values valid number? No NaN or inf
2. Are the values plausible? Ranges and units make sense: temperature around 200-300K, humidity around 0-1, pressure > 0, wind speed >= 0.
3. What do the distributions look like? For example, precipitation is very skewed with many dry days. This informs the type of transformations we might want to apply to the data.

In [1]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
xr.set_options(use_bottleneck=False)
# Disable bottleneck: xarray uses bottleneck to speed up 
# some operations (rolling etc.). But, it accumulates sums
# in float32, which gave wrong std value off by 1.9K.

In [2]:
# Data directory
DATADIR = Path('../data/lite/')

for path in sorted(DATADIR.glob('*.nc')):
    print(path.name)

NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_forcings_all.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_hist_train_1850_1931.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_hist_train_1932_2014.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_test_forcings_ssp245.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_train_ssp126.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_train_ssp370.nc
NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_train_ssp585.nc


In [3]:
filename = "NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_train_ssp126.nc"

ds = xr.open_dataset(DATADIR / filename)

In [4]:
tas = ds['tas']

p01 = tas.quantile(0.01).values
p50 = tas.quantile(0.50).values
p99 = tas.quantile(0.99).values

print("Mininum :", tas.min().values)
print("Maxinum :", tas.max().values)
print("Mean    :", tas.mean().values)
print("Std     :", tas.std().values)


print("1st percentile:", p01)
print("50st percentile:", p50)
print("99st percentile:", p99)

Mininum : 201.67216
Maxinum : 310.6852
Mean    : 278.92538
Std     : 21.091805
1st percentile: 219.1815020751953
50st percentile: 284.4742736816406
99st percentile: 303.3890380859375


In [5]:
tasmax = ds["tasmax"]
tasmin = ds["tasmin"]

violations = (tasmin > tas) | (tas > tasmax)

n_violations = violations.sum().item()

print("Total comparisons:", tas.size)
print("Temperature ordering violations:", n_violations)
print("Violation percentage:", 100 * n_violations / tas.size)

Total comparisons: 6780024
Temperature ordering violations: 0
Violation percentage: 0.0


The mean is not an area-weighted global mean. The data cover very cold and warm regions of Earth. The mean evaluation assigns equal weight to every grid cell. This is not a optimal representation because a sample near the pole will correspond a much smaller area than a sample near the equator. 

In [6]:
def summary_stats(ds, variables):
      """Pooled min/max/mean/std/percentiles for each variable in `variables` present in `ds`."""
      rows = []
      for name in variables:
          if name not in ds:          # skip variables this file doesn't have
              continue
          da = ds[name]
          rows.append({
              "variable": name,
              "units": da.attrs.get("units", "unknown"),
              "min": da.min().item(),
              "max": da.max().item(),
              "mean": da.mean().item(),
              "std": da.std().item(),
              "p01": da.quantile(0.01).item(),
              "median": da.quantile(0.50).item(),
              "p99": da.quantile(0.99).item(),
          })
      return pd.DataFrame(rows).set_index("variable")

In [7]:
targets = [
    "tas",
    "tasmax",
    "tasmin",
    "pr",
    "huss",
    "psl",
    "sfcWind"
]

summary_stats(ds, targets)

,units,min,max,mean,std,p01,median,p99
variable,,,,,,,,
tas,K,201.672165,310.685211,278.925385,21.091805,2.191815e+02,284.474274,303.389038
tasmax,K,203.783478,317.942566,280.799774,20.972507,2.209190e+02,286.270554,308.469452
tasmin,K,199.405441,304.934875,277.132507,21.351444,2.173769e+02,282.315399,301.616425
pr,kg m-2 s-1,0.000000,0.000553,0.000028,0.000034,6.335357e-08,0.000015,0.000156
huss,1,0.000003,0.023157,0.007793,0.006284,2.741040e-05,0.006105,0.020336
psl,Pa,92663.703125,106767.234375,100868.429688,1361.616455,9.692524e+04,101156.875000,103359.585938
sfcWind,m s-1,0.555534,21.830137,6.274629,2.530327,2.065205e+00,5.931022,13.461048


In [8]:
forcings = ['CO2_LBC', 'N2O_LBC', 'CH4_LBC',
            'CFC11eq_LBC', 'CF2CL2_LBC', 'BC_AX',
            'BC_N', 'SO2', 'SO4_PR', 'OM_NI']

summary_stats(ds, forcings)

,units,min,max,mean,std,p01,median,p99
variable,,,,,,,,
CO2_LBC,unknown,3.999489e-04,4.739992e-04,4.554925e-04,1.948150e-05,4.026418e-04,4.618903e-04,4.739988e-04
N2O_LBC,unknown,3.281798e-07,3.539130e-07,3.439978e-07,6.638315e-09,3.289415e-07,3.451634e-07,3.537327e-07
CH4_LBC,unknown,1.056409e-06,1.888389e-06,1.460188e-06,2.763243e-07,1.060489e-06,1.428844e-06,1.887463e-06
CFC11eq_LBC,unknown,3.159096e-10,8.891047e-10,5.716420e-10,2.070480e-10,3.179196e-10,5.086437e-10,8.885574e-10
CF2CL2_LBC,unknown,2.183210e-10,5.180193e-10,3.490634e-10,8.809364e-11,2.202368e-10,3.384335e-10,5.151000e-10
BC_AX,molecules cm-2 s-1,0.000000e+00,4.975203e+09,4.591731e+07,1.481893e+08,0.000000e+00,2.982460e+06,6.336111e+08
BC_N,molecules cm-2 s-1,0.000000e+00,4.477682e+10,4.132557e+08,1.333704e+09,0.000000e+00,2.684213e+07,5.702499e+09
SO2,molecules cm-2 s-1,0.000000e+00,3.116843e+10,2.968092e+08,9.024404e+08,0.000000e+00,4.526271e+07,4.254464e+09
SO4_PR,molecules cm-2 s-1,0.000000e+00,7.991906e+08,7.610492e+06,2.313950e+07,0.000000e+00,1.160582e+06,1.090888e+08
